In [3]:
from music21 import note, chord
import os
from collections import defaultdict
from music21 import converter


In [5]:
d = defaultdict(list)
d['apple']

[]

In [ ]:
score_path ="/Volumes/TOSHIBA  2T/CIPI_dataset/scores/xmander_files/559.musicxml"

In [3]:
def get_measure_rhythm(measure):
    """
    Returns a tuple of durations (quarterLength) for notes/rests in the measure.
    """
    rhythm = []
    for el in measure.notesAndRests:
        rhythm.append(el.duration.quarterLength)
    return tuple(rhythm)


In [4]:
def has_spillover(measure):
    for n in measure.notes:
        if n.tie and n.tie.type in ("start", "continue"):
            return True
    return False


In [41]:
score = '/Volumes/TOSHIBA  2T/CIPI_dataset/scores/xmander_files/29514.musicxml'
score = converter.parse(score).parts[0]
#measures = list(score.getElementsByClass('Measure'))

from music21 import quantize

q = quantize.Quantizer()
q.smallestDuration = 1/6
score = q.quantize(score)

ImportError: cannot import name 'quantize' from 'music21' (/Users/stepanpshenichnyi/occurence_dict/lib/python3.13/site-packages/music21/__init__.py)

In [38]:
notes = measures[0].notes

for n in notes:
    print(not n.tie)

True
True
True
True
True
True
True
True
True
True
True
True
True
True
True


In [ ]:
# Path to CIPI dataset
CIPI_PATH = "/Volumes/TOSHIBA  2T/CIPI_dataset/scores/xmander_files"

# Final dictionary
# meter -> pattern -> count
rhythm_dict = defaultdict(lambda: defaultdict(int))

for root, _, files in os.walk(CIPI_PATH):
    for file in files:
        if file.startswith("._"):
            continue
        
        if not file.lower().endswith((".xml", ".musicxml", ".mid", ".midi")):
            continue

        filepath = os.path.join(root, file)

        try:
            score = converter.parse(filepath)
        except Exception as e:
            print(f"Skipping {file}: {e}")
            continue

        # Use first part only (CIPI is mostly monophonic anyway)
        part = score.parts[0]

        # Get time signature
        ts = part.recurse().getElementsByClass('TimeSignature')
        if not ts:
            continue
        meter = ts[0].ratioString  # e.g. "2/4", "4/4"

        measures = list(part.getElementsByClass('Measure'))

        for i, m in enumerate(measures):
            pattern = get_measure_rhythm(m)
            if pattern:
                rhythm_dict[meter][pattern] += 1

            # Check for spillover → record 2-measure pattern
            if has_spillover(m) and i + 1 < len(measures):
                next_m = measures[i + 1]
                combined = pattern + get_measure_rhythm(next_m)
                rhythm_dict[meter][combined] += 1


In [15]:
GRID = 1/6

def quantize(d, grid):
    return round(d / grid) * grid

def is_ternary_group(durs, beat=1.0, tol=1e-6):
    """
    True if durations form a ternary subdivision of one beat.
    """
    if len(durs) != 3:
        return False
    if abs(sum(durs) - beat) > tol:
        return False
    return all(abs(d - beat/3) < tol for d in durs)



In [ ]:
def clean_rhythm(pattern, grid=1/6, beat=1.0):
    """
    Removes ornamental subdivisions but preserves structural ternary rhythms.
    """
    # Step 1: quantize
    q = [quantize(float(d), grid) for d in pattern]
    q = [d for d in q if d > 0]

    cleaned = []
    i = 0

    while i < len(q):
        # Check for ternary group
        if i + 2 < len(q):
            group = q[i:i+3]
            if is_ternary_group(group, beat):
                cleaned.extend(group)
                i += 3
                continue

        # Otherwise: collapse small ornamentation
        d = q[i]

        # If very small, merge into next
        if d < grid and i + 1 < len(q):
            q[i + 1] += d
        else:
            cleaned.append(d)

        i += 1

    return tuple(cleaned)


In [17]:
from collections import defaultdict

def postprocess_keep_ternary(rhythm_dict, grid=1/6):
    new_dict = defaultdict(lambda: defaultdict(int))

    for meter, patterns in rhythm_dict.items():
        for pattern, count in patterns.items():
            cleaned = clean_rhythm(pattern, grid)
            if cleaned:
                new_dict[meter][cleaned] += count

    return new_dict


In [18]:
processed = postprocess_keep_ternary(rhythm_dict)


In [20]:
import pickle

with open("cipi_rhythm_dict.pkl", "wb") as f:
    pickle.dump(dict(processed), f)
